Setup & Hardware Optimizations

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import timm
import numpy as np
import pandas as pd
from PIL import Image

from scripts.datasets import GeoguessrDataset
from scripts.model import GeoguessrModel
from scripts.losses import CoordinateLoss, haversine_distance, latlon_to_cartesian

# 1. HARDWARE OPTIMIZATION
torch.backends.cuda.matmul.allow_tf32 = True
cudnn.allow_tf32 = True
cudnn.benchmark = True  # SPEED 2: Auto-tune convolution algorithms
torch.set_float32_matmul_precision('medium')  # SPEED 3: Global TF32

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device} | TF32 Enabled: {torch.backends.cuda.matmul.allow_tf32} | cuDNN Benchmark: {cudnn.benchmark}")

os.makedirs("saved_models", exist_ok=True)

import cv2
cv2.setNumThreads(0)

# --- MASTER CONFIGURATION ---
BACKBONE_NAME = 'vit_base_patch16_siglip_256'
BATCH_SIZE = 16
NUM_WORKERS = 16 # Maxing out your 32-core Ryzen CPU!
LR_LAYER1 = 4e-5
LR_LAYER2 = 1e-4
# ----------------------------

Using device: cuda | TF32 Enabled: True | cuDNN Benchmark: True


The Core Training Engine

In [2]:
def get_dataset(dataset_type, folder=None):
    base_dir = os.path.abspath('.')
    if dataset_type == 'original':
        return GeoguessrDataset(csv_path=os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'clustered_training_data.csv'), 
                                image_dir=os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'images'), transform=None)
    elif dataset_type == 'extra':
        # FIX: Force Pandas to read 'folder' as a string so '00' doesn't become 0
        df = pd.read_csv(os.path.join(base_dir, 'extra_training_dataset', 'extra_clustered.csv'), dtype={'folder': str})
        # Double check formatting
        df['folder'] = df['folder'].str.zfill(2)
        
        subset_df = df[df['folder'] == folder].reset_index(drop=True)
        temp_csv = os.path.join(base_dir, 'extra_training_dataset', f'temp_{folder}.csv')
        subset_df.to_csv(temp_csv, index=False)
        return GeoguessrDataset(csv_path=temp_csv,
                                image_dir=os.path.join(base_dir, 'extra_training_dataset', 'images', folder), transform=None)

def set_trainable(model, train_backbone, train_layer1, train_layer2):
    for param in model.backbone.parameters(): param.requires_grad = train_backbone
    for param in model.country_head.parameters(): param.requires_grad = train_layer1
    for param in model.coordinate_head.parameters(): param.requires_grad = train_layer2

def train_epoch(model, dataloader, phase, uniform_probs=None, lr=1e-4):
    model.train()
    if phase == 'layer1':
        set_trainable(model, True, True, False)
        optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
        criterion = nn.CrossEntropyLoss()
    elif phase == 'layer2':
        set_trainable(model, False, False, True)
        optimizer = optim.AdamW(model.coordinate_head.parameters(), lr=lr)
        criterion = CoordinateLoss()
    elif phase == 'end_to_end':
        set_trainable(model, True, True, True)
        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': lr * 0.1},
            {'params': model.country_head.parameters(), 'lr': lr},
            {'params': model.coordinate_head.parameters(), 'lr': lr * 5}
        ])
        criterion_l1 = nn.CrossEntropyLoss()
        criterion_l2 = CoordinateLoss()
        
    scaler = torch.amp.GradScaler('cuda')
    progress_bar = tqdm(dataloader, desc=f"Training [{phase}]")
    
    for batch in progress_bar:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['country_label'].to(device, non_blocking=True)
        lats = batch['latitude'].to(device, non_blocking=True)
        lons = batch['longitude'].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            if phase == 'layer1':
                loss = criterion(model(images)['country_logits'], labels)
            elif phase == 'layer2':
                true_prob = 1.0 - (uniform_probs * (model.country_head.out_features - 1))
                force_probs = torch.full((images.size(0), model.country_head.out_features), uniform_probs, device=device)
                force_probs.scatter_(1, labels.unsqueeze(1), true_prob)
                loss = criterion(model(images, force_country_probs=force_probs)['pred_xyz'], lats, lons)
            elif phase == 'end_to_end':
                outputs = model(images)
                loss = criterion_l1(outputs['country_logits'], labels) + criterion_l2(outputs['pred_xyz'], lats, lons)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    # --- THE MODULAR SAVE LOGIC ---
    if phase in ['layer1', 'end_to_end']:
        torch.save({'backbone': model.backbone.state_dict(), 'country_head': model.country_head.state_dict()},
                   f"saved_models/layer1_{BACKBONE_NAME}.pth")
    if phase in ['layer2', 'end_to_end']:
        torch.save(model.coordinate_head.state_dict(), f"saved_models/layer2_{BACKBONE_NAME}.pth")
    print(f"--> Epoch saved to disk!")

original_dataset = get_dataset('original')
dummy_model = timm.create_model(BACKBONE_NAME, pretrained=False)
data_config = timm.data.resolve_data_config({}, model=dummy_model)
transform = A.Compose([A.Resize(data_config['input_size'][1], data_config['input_size'][1]),
                       A.Normalize(mean=data_config['mean'], std=data_config['std']), ToTensorV2()])
original_dataset.transform = transform
model = GeoguessrModel(num_countries=original_dataset.get_num_classes(), backbone_name=BACKBONE_NAME, pretrained=True).to(device)

path_l1 = f"saved_models/layer1_{BACKBONE_NAME}.pth"
path_l2 = f"saved_models/layer2_{BACKBONE_NAME}.pth"
if os.path.exists(path_l1):
    print(f"--> Found Layer 1 weights on disk. Resuming...")
    w1 = torch.load(path_l1, map_location=device, weights_only=True)
    model.backbone.load_state_dict(w1['backbone'], strict=False)
    model.country_head.load_state_dict(w1['country_head'], strict=False)
if os.path.exists(path_l2):
    print(f"--> Found Layer 2 weights on disk. Resuming...")
    model.coordinate_head.load_state_dict(torch.load(path_l2, map_location=device, weights_only=True), strict=False)

def train_folder(folder_id, uniform_probs):
    print(f"\n--- Loading Folder {folder_id} ---")
    dataset = get_dataset('extra', folder_id)
    dataset.transform = transform
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=True, prefetch_factor=2)
    train_epoch(model, loader, phase='layer1', lr=LR_LAYER1)
    train_epoch(model, loader, phase='layer2', uniform_probs=uniform_probs, lr=LR_LAYER2)
    del loader

--> Found Layer 1 weights on disk. Resuming...
--> Found Layer 2 weights on disk. Resuming...


Pass 1

In [3]:
train_folder('00', uniform_probs=0.002)


--- Loading Folder 00 ---


Training [layer1]: 100%|██████████| 3125/3125 [11:05<00:00,  4.70it/s, loss=4.4279] 


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:26<00:00, 21.29it/s, loss=0.0095]


--> Epoch saved to disk!


In [4]:
train_folder('01', uniform_probs=0.002)


--- Loading Folder 01 ---


Training [layer1]: 100%|██████████| 3125/3125 [09:06<00:00,  5.72it/s, loss=2.5037]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:24<00:00, 21.65it/s, loss=0.0132]


--> Epoch saved to disk!


In [5]:
train_folder('02', uniform_probs=0.002)


--- Loading Folder 02 ---


Training [layer1]: 100%|██████████| 3125/3125 [09:01<00:00,  5.77it/s, loss=2.4342]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:23<00:00, 21.78it/s, loss=0.0128]


--> Epoch saved to disk!


In [6]:
train_folder('03', uniform_probs=0.002)


--- Loading Folder 03 ---


Training [layer1]: 100%|██████████| 3125/3125 [08:51<00:00,  5.88it/s, loss=2.3647]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:23<00:00, 21.82it/s, loss=0.0081]


--> Epoch saved to disk!


Pass 2

In [4]:
LR_LAYER1 = 1e-5
LR_LAYER2 = 2.5e-5

In [8]:
train_folder('00', uniform_probs=0.001)


--- Loading Folder 00 ---


Training [layer1]: 100%|██████████| 3125/3125 [09:01<00:00,  5.77it/s, loss=1.6624]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:25<00:00, 21.52it/s, loss=0.0051]


--> Epoch saved to disk!


In [9]:
train_folder('01', uniform_probs=0.001)


--- Loading Folder 01 ---


Training [layer1]: 100%|██████████| 3125/3125 [09:15<00:00,  5.63it/s, loss=1.3090]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:24<00:00, 21.60it/s, loss=0.0349]


--> Epoch saved to disk!


In [10]:
train_folder('02', uniform_probs=0.001)


--- Loading Folder 02 ---


Training [layer1]: 100%|██████████| 3125/3125 [09:16<00:00,  5.62it/s, loss=0.8084] 


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:24<00:00, 21.60it/s, loss=0.0065]


--> Epoch saved to disk!


In [11]:
train_folder('03', uniform_probs=0.001)


--- Loading Folder 03 ---


Training [layer1]: 100%|██████████| 3125/3125 [09:05<00:00,  5.72it/s, loss=1.3753]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [02:20<00:00, 22.17it/s, loss=0.0052]


--> Epoch saved to disk!


Original Dataset Fine Tuning

In [12]:
LR_LAYER1 = 2e-5
LR_LAYER2 = 5e-5

print("\n[STEP 3] 5-EPOCH FINE-TUNING (Original 19k)")
loader_19k = DataLoader(original_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, persistent_workers=True, prefetch_factor=2)
for epoch in range(5):
    print(f"Epoch {epoch+1}/5")
    train_epoch(model, loader_19k, phase='end_to_end', lr=LR_LAYER1)


[STEP 3] 5-EPOCH FINE-TUNING (Original 19k)
Epoch 1/5


Training [end_to_end]: 100%|██████████| 1188/1188 [03:55<00:00,  5.03it/s, loss=4.8626] 


--> Epoch saved to disk!
Epoch 2/5


Training [end_to_end]: 100%|██████████| 1188/1188 [02:53<00:00,  6.84it/s, loss=4.8601]


--> Epoch saved to disk!
Epoch 3/5


...

--> Epoch saved to disk!
Epoch 4/5


Training [end_to_end]: 100%|██████████| 1188/1188 [02:52<00:00,  6.87it/s, loss=4.0245]


--> Epoch saved to disk!
Epoch 5/5


Training [end_to_end]: 100%|██████████| 1188/1188 [02:53<00:00,  6.84it/s, loss=3.6553]


--> Epoch saved to disk!


Cross validation

In [5]:
print("\n[STEP 4] CROSS-VALIDATION (Holdout 10k)")
holdout_dataset = get_dataset('extra', '04')
holdout_dataset.transform = transform
loader_10k_eval = DataLoader(holdout_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                             pin_memory=True, prefetch_factor=2)

model.eval()
all_distances = []
with torch.no_grad():
    for batch in tqdm(loader_10k_eval, desc="Evaluating (Folder 04)"):
        images, true_lats, true_lons = batch['image'].to(device), batch['latitude'].to(device), batch['longitude'].to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            distances = haversine_distance(outputs['pred_lat'], outputs['pred_lon'], true_lats, true_lons)
        all_distances.extend(distances.cpu().numpy().tolist())
print(f"--> Median Distance Error: {np.median(all_distances):.2f} km\n")


[STEP 4] CROSS-VALIDATION (Holdout 10k)


Evaluating (Folder 04): 100%|██████████| 633/633 [01:29<00:00,  7.05it/s] 

--> Median Distance Error: 2705.79 km



Incorporate Cross Validation Dataset

In [6]:
LR_LAYER1 = 1e-5
LR_LAYER2 = 2.5e-5

print("\n[STEP 5] TRAIN ON HOLDOUT (Folder 04)")
loader_10k_train = DataLoader(holdout_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              pin_memory=True, persistent_workers=True, prefetch_factor=2)
for epoch in range(2):
    print(f"Epoch {epoch+1}/2")
    train_epoch(model, loader_10k_train, phase='layer1', lr=LR_LAYER1)
    train_epoch(model, loader_10k_train, phase='layer2', uniform_probs=0.001, lr=LR_LAYER2)


[STEP 5] TRAIN ON HOLDOUT (Folder 04)
Epoch 1/2


Training [layer1]: 100%|██████████| 633/633 [02:32<00:00,  4.16it/s, loss=2.1296] 


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 633/633 [00:30<00:00, 20.99it/s, loss=0.0285]


--> Epoch saved to disk!
Epoch 2/2


Training [layer1]: 100%|██████████| 633/633 [01:32<00:00,  6.87it/s, loss=0.6256]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 633/633 [00:29<00:00, 21.24it/s, loss=0.0294]

--> Epoch saved to disk!


A Final Polish

In [3]:
LR_LAYER1 = 1e-5
LR_LAYER2 = 2.5e-5

print("\n[STEP 6] THE 3-EPOCH GOLDEN POLISH (Original 19k)")
loader_19k = DataLoader(original_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, persistent_workers=True, prefetch_factor=2)
for epoch in range(3):
    print(f"Epoch {epoch+1}/3")
    train_epoch(model, loader_19k, phase='end_to_end', lr=LR_LAYER1)

torch.save(model.state_dict(), f"saved_models/final_{BACKBONE_NAME}.pth")
print("Model Saved!")


[STEP 6] THE 3-EPOCH GOLDEN POLISH (Original 19k)
Epoch 1/3


Training [end_to_end]: 100%|██████████| 1188/1188 [03:30<00:00,  5.63it/s, loss=3.7736]


--> Epoch saved to disk!
Epoch 2/3


Training [end_to_end]: 100%|██████████| 1188/1188 [02:52<00:00,  6.87it/s, loss=3.6146]


--> Epoch saved to disk!
Epoch 3/3


Training [end_to_end]: 100%|██████████| 1188/1188 [02:53<00:00,  6.83it/s, loss=2.6884]


--> Epoch saved to disk!
Model Saved!


Submission

In [3]:
# 4. FINAL SUBMISSION GENERATION
from shapely.geometry import Point, shape
from shapely.ops import nearest_points
import json

class InferenceDataset(Dataset):
    def __init__(self, image_ids, test_image_dir, transform):
        self.image_ids = image_ids
        self.test_image_dir = test_image_dir
        self.transform = transform
        
    def __len__(self): 
        return len(self.image_ids)
        
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.test_image_dir, img_id)
        try:
            # 1. Safely open and close the file to prevent Windows memory leaks
            with Image.open(img_path) as img:
                image_np = np.array(img.convert('RGB'))
                
            # 2. Albumentations requires the "image=" keyword!
            tensor_img = self.transform(image=image_np)['image']
            return tensor_img, img_id, True
        except Exception as e:
            # Fallback for corrupted images
            return torch.zeros((3, 224, 224), dtype=torch.float32), img_id, False

def generate_final_submission(filename):
    print("\n" + "="*50 + "\nSTARTING FAST BATCH INFERENCE\n" + "="*50)
    model.eval()

    
    # 1. Load Country Polygons for Snapping
    with open('country_boundaries.geojson') as f:
        countries_data = json.load(f)
    country_polys = []
    for feature in countries_data['features']:
        try: country_polys.append(shape(feature['geometry']))
        except: continue
            
    def snap_to_land(lat, lon):
        point = Point(lon, lat)
        min_dist = float('inf')
        nearest_lat, nearest_lon = lat, lon
        for poly in country_polys:
            if poly.contains(point): return lat, lon
            dist = poly.distance(point)
            if dist < min_dist:
                min_dist = dist
                near_pt = nearest_points(point, poly)[1]
                nearest_lat, nearest_lon = near_pt.y, near_pt.x
        return nearest_lat, nearest_lon

    
    centroids_df = pd.read_csv('training_dataset/noised_dataset/cluster_centroids.csv')
    centroid_xyz = torch.tensor(centroids_df[['x', 'y', 'z']].values, dtype=torch.float32, device=device)
    centroid_indices = torch.tensor([original_dataset.label_mapping[cid] for cid in centroids_df['cluster_id']], device=device)
    
    df = pd.read_csv('submissions/sample_submission.csv')
    
    infer_loader = DataLoader(
        InferenceDataset(df['image_id'].tolist(), 'test_images_sampled', transform), 
        batch_size=64, 
        shuffle=False, 
        num_workers=0, 
        pin_memory=True
    )
    
    final_results = []
    with torch.no_grad():
        for images, image_ids, valids in tqdm(infer_loader, desc="Predicting Coordinates"):
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                pred_xyz = latlon_to_cartesian(outputs['pred_lat'], outputs['pred_lon'])
                
                dist_sq = torch.sum((pred_xyz.unsqueeze(1) - centroid_xyz.unsqueeze(0)) ** 2, dim=2)
                closest_idxs = torch.argmin(dist_sq, dim=1)
                target_classes = centroid_indices[closest_idxs]
                
                probs = F.softmax(outputs['country_logits'], dim=1)
                safe_probs = probs[torch.arange(images.size(0), device=device), target_classes].cpu().numpy()
                
            lats_np, lons_np = outputs['pred_lat'].cpu().numpy(), outputs['pred_lon'].cpu().numpy()
            
            for i in range(len(image_ids)):
                img_id = image_ids[i]
                if valids[i]:
                    final_lat, final_lon = snap_to_land(lats_np[i], lons_np[i])
                    radius = max(64.0, min(6400.0, 6400 * np.exp(-4.6 * safe_probs[i])))
                    
                    final_results.append({'image_id': img_id, 'pred_lat': final_lat, 'pred_lon': final_lon,
                                          'pred_radius_km': radius})
                else:
                    final_results.append({'image_id': img_id, 'pred_lat': 0.0, 'pred_lon': 0.0, 'pred_radius_km': 6400.0})

    out_path = f"submissions/{filename}.csv"
    pd.DataFrame(final_results).set_index('image_id').reindex(df['image_id']).reset_index().to_csv(out_path, index=False)
    print(f"SUCCESS! Submission saved to: {os.path.abspath(out_path)}")

In [5]:
generate_final_submission("siglip256_phase1")


STARTING FAST BATCH INFERENCE


Predicting Coordinates: 100%|██████████| 8/8 [00:07<00:00,  1.06it/s]

SUCCESS! Submission saved to: c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\submissions\siglip256_phase1.csv


Data Augmentation

In [4]:
# 1. Define the dynamic augmentation pipelines
req_size = data_config['input_size'][1]
base_norm = A.Normalize(mean=data_config['mean'], std=data_config['std'])

# Pipeline A: Light Color Jitter (We keep hue shift very low to protect dirt/sky colors)
jitter_transform = A.Compose([
    A.Resize(req_size, req_size),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=1.0),
    base_norm,
    ToTensorV2()
])

# Pipeline B: Light Image Compression (Simulates old Gen 1/2 Google Cameras)
compress_transform = A.Compose([
    A.Resize(req_size, req_size),
    A.ImageCompression(quality_lower=80, quality_upper=95, p=1.0),
    base_norm,
    ToTensorV2()
])

# Pipeline C: Pristine / Clean (The exact Hackathon distribution)
clean_transform = A.Compose([
    A.Resize(req_size, req_size),
    base_norm,
    ToTensorV2()
])

C:\Users\Yash T\AppData\Local\Temp\ipykernel_11140\1841987428.py:16: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=80, quality_upper=95, p=1.0),


4 Epoch Polish Using Augmented Data

In [5]:
LR_LAYER1 = 1e-5
LR_LAYER2 = 2.5e-5

print("\n[STEP 6] THE 4-EPOCH GOLDEN POLISH (With Dynamic Augmentation)")

for epoch in range(1, 5):
    # Dynamically swap the physical augmentation pipeline
    if epoch in [1, 2]:
        print(f"\n--- Epoch {epoch}/4 [Augmentation: Light Color Jitter] ---")
        original_dataset.transform = jitter_transform
    elif epoch == 3:
        print(f"\n--- Epoch {epoch}/4 [Augmentation: Light Image Compression] ---")
        original_dataset.transform = compress_transform
    elif epoch == 4:
        print(f"\n--- Epoch {epoch}/4 [Augmentation: Pristine / Original] ---")
        original_dataset.transform = clean_transform
        
    # We recreate the DataLoader so the 16 background workers instantly pick up the new transform rules!
    loader_19k_aug = DataLoader(original_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                                pin_memory=True, prefetch_factor=2)
    
    # Train End-to-End!
    train_epoch(model, loader_19k_aug, phase='end_to_end', lr=LR_LAYER1)


print("\n[SAVING FINAL MODEL]")
torch.save(model.state_dict(), f"saved_models/final_{BACKBONE_NAME}.pth")
print("Master Curriculum Execution Complete!")


[STEP 6] THE 4-EPOCH GOLDEN POLISH (With Dynamic Augmentation)

--- Epoch 1/4 [Augmentation: Light Color Jitter] ---


Training [end_to_end]:   0%|          | 0/1188 [00:32<?, ?it/s]


KeyboardInterrupt: 

Pass 3

In [ ]:
LR_LAYER1 = 1e-5
LR_LAYER2 = 2.5e-5

train_folder('00', uniform_probs=0.002)
train_folder('01', uniform_probs=0.002)
train_folder('02', uniform_probs=0.002)
train_folder('03', uniform_probs=0.002)
train_folder('04', uniform_probs=0.002)


--- Loading Folder 00 ---


Training [layer1]: 100%|██████████| 3125/3125 [14:47<00:00,  3.52it/s, loss=0.1246]


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 3125/3125 [03:39<00:00, 14.22it/s, loss=0.0055]


--> Epoch saved to disk!

--- Loading Folder 01 ---


...

--> Epoch saved to disk!

--- Loading Folder 04 ---


Training [layer1]: 100%|██████████| 633/633 [03:35<00:00,  2.94it/s, loss=0.0559] 


--> Epoch saved to disk!


Training [layer2]: 100%|██████████| 633/633 [00:44<00:00, 14.34it/s, loss=0.0078]


--> Epoch saved to disk!


In [ ]:
loader_19k = DataLoader(original_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, persistent_workers=True, prefetch_factor=2)
for epoch in range(3):
    print(f"Epoch {epoch+1}/3")
    train_epoch(model, loader_19k, phase='end_to_end', lr=LR_LAYER1)

torch.save(model.state_dict(), f"saved_models/final_{BACKBONE_NAME}.pth")
print("Model Saved!")

Epoch 1/3


Training [end_to_end]: 100%|██████████| 1188/1188 [05:54<00:00,  3.35it/s, loss=3.1876]


--> Epoch saved to disk!
Epoch 2/3


Training [end_to_end]: 100%|██████████| 1188/1188 [05:12<00:00,  3.81it/s, loss=2.5915]


--> Epoch saved to disk!
Epoch 3/3


Training [end_to_end]: 100%|██████████| 1188/1188 [05:04<00:00,  3.90it/s, loss=2.2704]


--> Epoch saved to disk!
Model Saved!


In [ ]:
generate_final_submission("siglip256_phase2")

Evaluate

In [ ]:
def evaluate_original_dataset():
    print("\n[EVALUATION] Checking performance on the Original 19k Dataset...")
    
    # 1. Force the dataset to use the pristine/clean transform for evaluation!
    original_dataset.transform = clean_transform
    
    # 2. Create the evaluation loader (shuffle=False is slightly faster for eval)
    eval_loader = DataLoader(
        original_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS, 
        pin_memory=True,
        prefetch_factor=2
    )
    
    model.eval()
    all_distances = []
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="Evaluating Original Data"):
            images = batch['image'].to(device, non_blocking=True)
            true_lats = batch['latitude'].to(device, non_blocking=True)
            true_lons = batch['longitude'].to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                # Calculate the exact real-world kilometer distance between prediction and ground truth
                distances = haversine_distance(outputs['pred_lat'], outputs['pred_lon'], true_lats, true_lons)
                
            all_distances.extend(distances.cpu().numpy().tolist())
            
    median_error = np.median(all_distances)
    mean_error = np.mean(all_distances)
    
    print("\n" + "="*40)
    print(f"--> Median Distance Error: {median_error:.2f} km")
    print(f"--> Mean Distance Error:   {mean_error:.2f} km")
    print("="*40 + "\n")

In [ ]:
evaluate_original_dataset()


[EVALUATION] Checking performance on the Original 19k Dataset...


Evaluating Original Data: 100%|██████████| 1188/1188 [02:35<00:00,  7.63it/s]


--> Median Distance Error: 2205.11 km
--> Mean Distance Error:   3312.47 km

